In [1]:
# CELL 1: Day 2 - Setup and Environment


import os
import sys
import warnings
import subprocess
import importlib.util
from datetime import datetime

# Suppress warnings for cleaner output
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

print("DAY 2: Planning and Agent Memory")
print("=" * 60)
print(f"Started at: {datetime.now().isoformat()}")

# List of required packages for Day 2
required_packages = [
    ("langgraph", "langgraph"),
    ("langchain", "langchain"),
    ("langchain_core", "langchain-core"),
    ("langchain_community", "langchain-community"),
    ("wikipediaapi", "wikipedia-api"),
    ("requests", "requests"),
    ("pydantic", "pydantic"),
    ("typing_extensions", "typing_extensions"),
]

def install_package(package_name):
    """Install a package using pip with quiet output."""
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package_name],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        return True
    except subprocess.CalledProcessError:
        return False

def check_and_install_packages():
    """Check if packages are installed, install missing ones."""
    installed = []
    failed = []
    
    for import_name, pip_name in required_packages:
        spec = importlib.util.find_spec(import_name)
        if spec is None:
            print(f"Installing: {pip_name}")
            if install_package(pip_name):
                installed.append(pip_name)
            else:
                failed.append(pip_name)
        else:
            print(f"Already installed: {pip_name}")
    
    if installed:
        print(f"\nSuccessfully installed: {', '.join(installed)}")
    if failed:
        print(f"\nFailed to install: {', '.join(failed)}")
    
    return len(failed) == 0

print("\nChecking and installing required packages...")
print("-" * 40)
installation_success = check_and_install_packages()
print("-" * 40)

if installation_success:
    print("All packages are ready.")
else:
    print("Some packages failed to install.")

print("\n" + "=" * 60)
print("Day 2 environment setup complete.")

DAY 2: Planning and Agent Memory
Started at: 2026-09-01T06:33:32.197637

Checking and installing required packages...
----------------------------------------
Already installed: langgraph
Already installed: langchain
Already installed: langchain-core
Installing: langchain-community
Installing: wikipedia-api
Already installed: requests
Already installed: pydantic
Already installed: typing_extensions

----------------------------------------
All packages are ready.

Day 2 environment setup complete.


In [2]:
# CELL 2: Task Planner - Task Decomposition System
# This cell creates a task planner that breaks complex queries into subtasks

from typing import Dict, Any, Optional, List, Tuple
from datetime import datetime
import re
import json

class Task:
    """
    Represents a single task in the planning system.
    """
    
    def __init__(self, task_id: str, description: str, tool: str = None, 
                 priority: int = 1, dependencies: List[str] = None):
        """
        Initialize a task.
        
        Args:
            task_id: Unique identifier for the task
            description: Human-readable description of the task
            tool: The tool to use for this task (optional)
            priority: Priority level (1 = highest)
            dependencies: List of task IDs that must complete before this task
        """
        self.task_id = task_id
        self.description = description
        self.tool = tool
        self.priority = priority
        self.dependencies = dependencies or []
        self.status = "pending"  # pending, running, completed, failed
        self.result = None
        self.error = None
        self.created_at = datetime.now().isoformat()
        self.completed_at = None
    
    def to_dict(self) -> Dict[str, Any]:
        """Convert task to dictionary."""
        return {
            "task_id": self.task_id,
            "description": self.description,
            "tool": self.tool,
            "priority": self.priority,
            "dependencies": self.dependencies,
            "status": self.status,
            "result": self.result,
            "error": self.error,
            "created_at": self.created_at,
            "completed_at": self.completed_at
        }
    
    def mark_completed(self, result: Any) -> None:
        """Mark task as completed with result."""
        self.status = "completed"
        self.result = result
        self.completed_at = datetime.now().isoformat()
    
    def mark_failed(self, error: str) -> None:
        """Mark task as failed."""
        self.status = "failed"
        self.error = error
        self.completed_at = datetime.now().isoformat()

class TaskPlanner:
    """
    Task planner that decomposes complex queries into manageable subtasks.
    """
    
    def __init__(self, available_tools: List[str] = None):
        """
        Initialize the task planner.
        
        Args:
            available_tools: List of tool names available for tasks
        """
        self.available_tools = available_tools or ["wikipedia_search", "web_search", "calculator"]
        self.task_history: List[Dict[str, Any]] = []
        self.task_counter = 0
    
    def _generate_task_id(self) -> str:
        """Generate a unique task ID."""
        self.task_counter += 1
        return f"task_{self.task_counter:03d}"
    
    def _extract_calculations(self, query: str) -> List[Dict[str, Any]]:
        """
        Extract calculation needs from the query.
        
        Args:
            query: The user query
            
        Returns:
            List of calculation tasks
        """
        calc_tasks = []
        
        # Patterns for mathematical expressions
        patterns = [
            r'calculate\s+([\d+\-*/()\s.^]+)',
            r'what is\s+([\d+\-*/()\s.^]+)',
            r'compute\s+([\d+\-*/()\s.^]+)',
            r'(\d+\s*[\+\-\*/]\s*\d+)',
            r'(\d+\s*[\+\-\*/]\s*\d+\s*[\+\-\*/]\s*\d+)'
        ]
        
        for pattern in patterns:
            matches = re.findall(pattern, query.lower())
            for match in matches:
                expr = match.strip()
                if expr and re.search(r'[\d+\-*/]', expr):
                    task_id = self._generate_task_id()
                    calc_tasks.append({
                        "task_id": task_id,
                        "description": f"Calculate: {expr}",
                        "tool": "calculator",
                        "priority": 2,
                        "dependencies": [],
                        "expression": expr
                    })
        
        return calc_tasks
    
    def _extract_research_needs(self, query: str) -> List[Dict[str, Any]]:
        """
        Extract research needs from the query.
        
        Args:
            query: The user query
            
        Returns:
            List of research tasks
        """
        research_tasks = []
        
        # Extract topics for Wikipedia search
        topics = []
        
        # Pattern for "what is X" or "who is X"
        what_patterns = [
            r'what is\s+([a-zA-Z\s]+)',
            r'who is\s+([a-zA-Z\s]+)',
            r'define\s+([a-zA-Z\s]+)',
            r'about\s+([a-zA-Z\s]+)',
            r'research\s+([a-zA-Z\s]+)'
        ]
        
        for pattern in what_patterns:
            matches = re.findall(pattern, query.lower())
            for match in matches:
                topic = match.strip()
                if topic and len(topic) > 2 and topic not in topics:
                    topics.append(topic)
        
        # If no specific topics found, extract keywords
        if not topics:
            words = query.split()
            stopwords = {'what', 'is', 'are', 'the', 'of', 'and', 'to', 'for', 'in', 'on', 
                        'at', 'with', 'by', 'about', 'research', 'find', 'search', 'tell', 'me'}
            keywords = [w for w in words if w.lower() not in stopwords and len(w) > 3]
            if keywords:
                topics.append(' '.join(keywords[:2]))
        
        # Create research tasks
        for topic in topics:
            task_id = self._generate_task_id()
            research_tasks.append({
                "task_id": task_id,
                "description": f"Research: {topic}",
                "tool": "wikipedia_search",
                "priority": 1,
                "dependencies": [],
                "query": topic
            })
        
        return research_tasks
    
    def _extract_general_search(self, query: str) -> List[Dict[str, Any]]:
        """
        Extract general web search needs.
        
        Args:
            query: The user query
            
        Returns:
            List of search tasks
        """
        search_tasks = []
        
        # Check for search keywords
        search_keywords = ['search', 'find', 'look up', 'google', 'web', 'online']
        if any(kw in query.lower() for kw in search_keywords):
            task_id = self._generate_task_id()
            search_tasks.append({
                "task_id": task_id,
                "description": f"Web search: {query[:50]}",
                "tool": "web_search",
                "priority": 3,
                "dependencies": [],
                "query": query
            })
        
        return search_tasks
    
    def create_plan(self, query: str) -> Dict[str, Any]:
        """
        Create a plan by decomposing the query into tasks.
        
        Args:
            query: The user query
            
        Returns:
            Dictionary containing the plan with tasks
        """
        print(f"\nPlanning for query: {query}")
        print("-" * 40)
        
        # Extract different types of tasks
        all_tasks = []
        all_tasks.extend(self._extract_research_needs(query))
        all_tasks.extend(self._extract_calculations(query))
        all_tasks.extend(self._extract_general_search(query))
        
        # If no tasks were identified, create a default search task
        if not all_tasks:
            task_id = self._generate_task_id()
            all_tasks.append({
                "task_id": task_id,
                "description": f"Search for: {query}",
                "tool": "web_search",
                "priority": 1,
                "dependencies": [],
                "query": query
            })
        
        # Sort by priority (lower number = higher priority)
        all_tasks.sort(key=lambda x: x['priority'])
        
        # Create Task objects
        task_objects = []
        for task_data in all_tasks:
            task = Task(
                task_id=task_data['task_id'],
                description=task_data['description'],
                tool=task_data['tool'],
                priority=task_data['priority'],
                dependencies=task_data.get('dependencies', [])
            )
            # Store additional data
            if 'expression' in task_data:
                task.expression = task_data['expression']
            if 'query' in task_data:
                task.search_query = task_data['query']
            task_objects.append(task)
        
        # Create plan
        plan = {
            "original_query": query,
            "created_at": datetime.now().isoformat(),
            "total_tasks": len(task_objects),
            "tasks": [t.to_dict() for t in task_objects],
            "task_objects": task_objects,
            "execution_order": [t.task_id for t in sorted(task_objects, key=lambda x: (x.priority, len(x.dependencies)))]
        }
        
        # Store in history
        self.task_history.append(plan)
        
        # Print plan summary
        print(f"Created plan with {len(task_objects)} tasks:")
        for task in task_objects:
            deps = f" (depends on: {', '.join(task.dependencies)})" if task.dependencies else ""
            print(f"  - {task.task_id}: {task.description} [{task.tool}]{deps}")
        
        return plan
    
    def get_next_task(self, plan: Dict[str, Any]) -> Optional[Task]:
        """
        Get the next executable task from a plan.
        
        Args:
            plan: The plan dictionary
            
        Returns:
            Next task or None if no tasks available
        """
        task_objects = plan.get('task_objects', [])
        
        for task in task_objects:
            if task.status == "pending":
                # Check if dependencies are met
                dependencies_met = True
                for dep_id in task.dependencies:
                    dep_task = next((t for t in task_objects if t.task_id == dep_id), None)
                    if dep_task and dep_task.status != "completed":
                        dependencies_met = False
                        break
                
                if dependencies_met:
                    return task
        
        return None
    
    def get_task_status_summary(self, plan: Dict[str, Any]) -> Dict[str, int]:
        """
        Get a summary of task statuses for a plan.
        
        Args:
            plan: The plan dictionary
            
        Returns:
            Dictionary with status counts
        """
        task_objects = plan.get('task_objects', [])
        status_counts = {"pending": 0, "running": 0, "completed": 0, "failed": 0}
        
        for task in task_objects:
            status_counts[task.status] = status_counts.get(task.status, 0) + 1
        
        return status_counts
    
    def get_history(self) -> List[Dict[str, Any]]:
        """Get planning history."""
        return self.task_history

def create_task_planner(available_tools: List[str] = None) -> TaskPlanner:
    """Factory function to create a task planner."""
    return TaskPlanner(available_tools)

# Test the task planner
print("Testing Task Planner...")
print("=" * 60)

planner = create_task_planner()

# Test queries
test_queries = [
    "What is artificial intelligence and calculate 100/4",
    "Research Python programming and compute 15*3",
    "Find information about machine learning",
    "What is quantum computing? Also what is 2+2"
]

for query in test_queries:
    print("\n" + "=" * 40)
    plan = planner.create_plan(query)
    print(f"\nExecution order: {plan['execution_order']}")
    print(f"Total tasks: {plan['total_tasks']}")
    
    # Show status summary
    status = planner.get_task_status_summary(plan)
    print(f"Status: {status}")

print("\n" + "=" * 60)
print("Task Planner test completed.")

Testing Task Planner...


Planning for query: What is artificial intelligence and calculate 100/4
----------------------------------------
Created plan with 3 tasks:
  - task_001: Research: artificial intelligence and calculate [wikipedia_search]
  - task_002: Calculate: 100/4 [calculator]
  - task_003: Calculate: 100/4 [calculator]

Execution order: ['task_001', 'task_002', 'task_003']
Total tasks: 3
Status: {'pending': 3, 'running': 0, 'completed': 0, 'failed': 0}


Planning for query: Research Python programming and compute 15*3
----------------------------------------
Created plan with 4 tasks:
  - task_004: Research: python programming and compute [wikipedia_search]
  - task_005: Calculate: 15*3 [calculator]
  - task_006: Calculate: 15*3 [calculator]
  - task_007: Web search: Research Python programming and compute 15*3 [web_search]

Execution order: ['task_004', 'task_005', 'task_006', 'task_007']
Total tasks: 4
Status: {'pending': 4, 'running': 0, 'completed': 0, 'failed': 0}




In [3]:
# CELL 3: Memory System - Short-term, Long-term, and Episodic Memory
# This cell creates a comprehensive memory system for the agent

from typing import Dict, Any, Optional, List, Tuple
from datetime import datetime
import json
import hashlib
import re

class MemoryEntry:
    """
    Represents a single memory entry.
    """
    
    def __init__(self, content: str, memory_type: str = "episodic", 
                 importance: int = 1, metadata: Dict[str, Any] = None):
        """
        Initialize a memory entry.
        
        Args:
            content: The memory content
            memory_type: Type of memory (short_term, long_term, episodic)
            importance: Importance score (1-10)
            metadata: Additional metadata
        """
        self.entry_id = self._generate_id(content)
        self.content = content
        self.memory_type = memory_type
        self.importance = importance
        self.metadata = metadata or {}
        self.created_at = datetime.now().isoformat()
        self.last_accessed = self.created_at
        self.access_count = 0
    
    def _generate_id(self, content: str) -> str:
        """Generate a unique ID for the memory entry."""
        return hashlib.md5(content.encode()).hexdigest()[:8]
    
    def access(self) -> None:
        """Update access statistics."""
        self.access_count += 1
        self.last_accessed = datetime.now().isoformat()
    
    def to_dict(self) -> Dict[str, Any]:
        """Convert memory entry to dictionary."""
        return {
            "entry_id": self.entry_id,
            "content": self.content,
            "memory_type": self.memory_type,
            "importance": self.importance,
            "metadata": self.metadata,
            "created_at": self.created_at,
            "last_accessed": self.last_accessed,
            "access_count": self.access_count
        }

class MemorySystem:
    """
    Comprehensive memory system with short-term, long-term, and episodic memory.
    """
    
    def __init__(self, max_short_term: int = 10, max_long_term: int = 100):
        """
        Initialize the memory system.
        
        Args:
            max_short_term: Maximum number of short-term memories
            max_long_term: Maximum number of long-term memories
        """
        self.short_term_memory: List[MemoryEntry] = []
        self.long_term_memory: List[MemoryEntry] = []
        self.episodic_memory: List[MemoryEntry] = []
        self.max_short_term = max_short_term
        self.max_long_term = max_long_term
        self.context_window: List[Dict[str, Any]] = []
        
        print(f"Memory System initialized:")
        print(f"  - Short-term capacity: {max_short_term}")
        print(f"  - Long-term capacity: {max_long_term}")
    
    def store(self, content: str, memory_type: str = "short_term", 
              importance: int = 1, metadata: Dict[str, Any] = None) -> str:
        """
        Store a memory entry.
        
        Args:
            content: The content to store
            memory_type: Type of memory (short_term, long_term, episodic)
            importance: Importance score (1-10)
            metadata: Additional metadata
            
        Returns:
            The entry ID
        """
        entry = MemoryEntry(content, memory_type, importance, metadata)
        
        if memory_type == "short_term":
            self.short_term_memory.append(entry)
            # Enforce capacity
            if len(self.short_term_memory) > self.max_short_term:
                # Remove least important entries
                self.short_term_memory.sort(key=lambda x: (x.importance, x.access_count))
                removed = self.short_term_memory.pop(0)
                print(f"  Removed from short-term: {removed.content[:50]}...")
        
        elif memory_type == "long_term":
            self.long_term_memory.append(entry)
            if len(self.long_term_memory) > self.max_long_term:
                # Remove oldest entries
                self.long_term_memory.sort(key=lambda x: x.created_at)
                removed = self.long_term_memory.pop(0)
                print(f"  Removed from long-term: {removed.content[:50]}...")
        
        elif memory_type == "episodic":
            self.episodic_memory.append(entry)
        
        # Also add to context window
        self.context_window.append({
            "type": memory_type,
            "content": content,
            "importance": importance,
            "timestamp": entry.created_at
        })
        
        # Trim context window
        if len(self.context_window) > 20:
            self.context_window = self.context_window[-20:]
        
        print(f"  Stored in {memory_type}: {content[:50]}...")
        return entry.entry_id
    
    def retrieve(self, query: str, memory_type: str = None, 
                 limit: int = 5) -> List[MemoryEntry]:
        """
        Retrieve memories matching a query.
        
        Args:
            query: Search query
            memory_type: Filter by memory type (None = all)
            limit: Maximum results
            
        Returns:
            List of matching memory entries
        """
        results = []
        query_lower = query.lower()
        
        # Select memory pools
        pools = []
        if memory_type is None or memory_type == "short_term":
            pools.append(("short_term", self.short_term_memory))
        if memory_type is None or memory_type == "long_term":
            pools.append(("long_term", self.long_term_memory))
        if memory_type is None or memory_type == "episodic":
            pools.append(("episodic", self.episodic_memory))
        
        # Search each pool
        for pool_name, pool in pools:
            for entry in pool:
                # Simple keyword matching
                if query_lower in entry.content.lower():
                    entry.access()
                    results.append(entry)
        
        # Sort by relevance (access count and importance)
        results.sort(key=lambda x: (x.importance, x.access_count), reverse=True)
        
        # Update access statistics for retrieved entries
        for entry in results[:limit]:
            entry.access()
        
        return results[:limit]
    
    def get_recent_context(self, limit: int = 5) -> List[Dict[str, Any]]:
        """
        Get recent context from the context window.
        
        Args:
            limit: Number of recent entries
            
        Returns:
            List of recent context entries
        """
        return self.context_window[-limit:]
    
    def summarize_memories(self) -> Dict[str, Any]:
        """
        Get a summary of all memories.
        
        Returns:
            Dictionary with memory statistics
        """
        return {
            "short_term_count": len(self.short_term_memory),
            "long_term_count": len(self.long_term_memory),
            "episodic_count": len(self.episodic_memory),
            "total_memories": len(self.short_term_memory) + len(self.long_term_memory) + len(self.episodic_memory),
            "context_window_size": len(self.context_window)
        }
    
    def clear_short_term(self) -> None:
        """Clear short-term memory."""
        self.short_term_memory.clear()
        print("Short-term memory cleared.")
    
    def clear_all(self) -> None:
        """Clear all memories."""
        self.short_term_memory.clear()
        self.long_term_memory.clear()
        self.episodic_memory.clear()
        self.context_window.clear()
        print("All memories cleared.")
    
    def get_all_memories(self) -> Dict[str, List[Dict[str, Any]]]:
        """
        Get all memories as dictionaries.
        
        Returns:
            Dictionary with all memory types
        """
        return {
            "short_term": [e.to_dict() for e in self.short_term_memory],
            "long_term": [e.to_dict() for e in self.long_term_memory],
            "episodic": [e.to_dict() for e in self.episodic_memory]
        }

def create_memory_system(max_short_term: int = 10, max_long_term: int = 100) -> MemorySystem:
    """Factory function to create a memory system."""
    return MemorySystem(max_short_term, max_long_term)

# Test the memory system
print("Testing Memory System...")
print("=" * 60)

memory = create_memory_system()

print("\nStoring memories:")
memory.store("Artificial Intelligence is the simulation of human intelligence in machines", "short_term", 8)
memory.store("Machine Learning is a subset of AI that uses algorithms", "short_term", 7)
memory.store("Python is a high-level programming language", "long_term", 6)
memory.store("The user asked about quantum computing earlier", "episodic", 5, {"conversation": "day2"})
memory.store("Neural networks are inspired by the human brain", "long_term", 7)

print("\nRetrieving memories:")
results = memory.retrieve("AI", limit=3)
for entry in results:
    print(f"  - {entry.content[:80]}... (type: {entry.memory_type}, importance: {entry.importance})")

print("\nRecent context:")
context = memory.get_recent_context(3)
for item in context:
    print(f"  - {item['type']}: {item['content'][:50]}...")

print("\nMemory summary:")
summary = memory.summarize_memories()
for key, value in summary.items():
    print(f"  - {key}: {value}")

print("\n" + "=" * 60)
print("Memory System test completed.")

Testing Memory System...
Memory System initialized:
  - Short-term capacity: 10
  - Long-term capacity: 100

Storing memories:
  Stored in short_term: Artificial Intelligence is the simulation of human...
  Stored in short_term: Machine Learning is a subset of AI that uses algor...
  Stored in long_term: Python is a high-level programming language...
  Stored in episodic: The user asked about quantum computing earlier...
  Stored in long_term: Neural networks are inspired by the human brain...

Retrieving memories:
  - Machine Learning is a subset of AI that uses algorithms... (type: short_term, importance: 7)
  - Neural networks are inspired by the human brain... (type: long_term, importance: 7)

Recent context:
  - long_term: Python is a high-level programming language...
  - episodic: The user asked about quantum computing earlier...
  - long_term: Neural networks are inspired by the human brain...

Memory summary:
  - short_term_count: 2
  - long_term_count: 2
  - episodic_count: 1

In [4]:
# CELL 4: LangGraph State Management
# This cell creates the state management system using LangGraph

from typing import Dict, Any, Optional, List, Tuple, Literal, Annotated
from datetime import datetime
import json
import operator

# Try to import LangGraph
try:
    from langgraph.graph import StateGraph, END
    from langgraph.graph.message import add_messages
    from typing import TypedDict
    LANGGRAPH_AVAILABLE = True
    print("LangGraph imported successfully.")
except ImportError:
    LANGGRAPH_AVAILABLE = False
    print("LangGraph not available. Using custom state management.")

class AgentState:
    """
    State management for the agent.
    Tracks the current state, plan, execution results, and memory.
    """
    
    def __init__(self):
        self.query = ""
        self.plan = None
        self.current_task_index = 0
        self.results = {}
        self.errors = {}
        self.status = "initialized"  # initialized, planning, executing, reviewing, completed, failed
        self.memory_context = []
        self.execution_history = []
        self.start_time = None
        self.end_time = None
        self.final_response = ""
    
    def to_dict(self) -> Dict[str, Any]:
        """Convert state to dictionary."""
        return {
            "query": self.query,
            "plan": self.plan,
            "current_task_index": self.current_task_index,
            "results": self.results,
            "errors": self.errors,
            "status": self.status,
            "memory_context": self.memory_context,
            "execution_history": self.execution_history,
            "start_time": self.start_time,
            "end_time": self.end_time,
            "final_response": self.final_response
        }
    
    def update(self, **kwargs) -> None:
        """Update state attributes."""
        for key, value in kwargs.items():
            if hasattr(self, key):
                setattr(self, key, value)
    
    def add_result(self, task_id: str, result: Any) -> None:
        """Add a task result."""
        self.results[task_id] = result
        self.execution_history.append({
            "task_id": task_id,
            "result": result,
            "timestamp": datetime.now().isoformat()
        })
    
    def add_error(self, task_id: str, error: str) -> None:
        """Add a task error."""
        self.errors[task_id] = error
        self.execution_history.append({
            "task_id": task_id,
            "error": error,
            "timestamp": datetime.now().isoformat()
        })
    
    def add_memory_context(self, memory_entry: Dict[str, Any]) -> None:
        """Add memory context to state."""
        self.memory_context.append(memory_entry)
    
    def is_complete(self) -> bool:
        """Check if all tasks are complete."""
        if not self.plan:
            return False
        tasks = self.plan.get('task_objects', [])
        if not tasks:
            return True
        completed = sum(1 for t in tasks if t.status == "completed")
        return completed == len(tasks)
    
    def get_next_task(self):
        """Get the next pending task."""
        if not self.plan:
            return None
        tasks = self.plan.get('task_objects', [])
        for task in tasks:
            if task.status == "pending":
                # Check dependencies
                deps_met = True
                for dep_id in task.dependencies:
                    dep_task = next((t for t in tasks if t.task_id == dep_id), None)
                    if dep_task and dep_task.status != "completed":
                        deps_met = False
                        break
                if deps_met:
                    return task
        return None
    
    def get_task_summary(self) -> Dict[str, int]:
        """Get summary of task statuses."""
        if not self.plan:
            return {}
        tasks = self.plan.get('task_objects', [])
        summary = {"pending": 0, "running": 0, "completed": 0, "failed": 0}
        for task in tasks:
            summary[task.status] = summary.get(task.status, 0) + 1
        return summary

class StateManager:
    """
    Manages agent state and transitions between states.
    """
    
    def __init__(self):
        self.state = AgentState()
        self.state_history: List[Dict[str, Any]] = []
        self.transitions: List[Dict[str, Any]] = []
    
    def initialize(self, query: str) -> None:
        """Initialize state with a query."""
        self.state.query = query
        self.state.status = "initialized"
        self.state.start_time = datetime.now().isoformat()
        self._save_state("initialize")
        print(f"State initialized for query: {query}")
    
    def set_plan(self, plan: Dict[str, Any]) -> None:
        """Set the plan in state."""
        self.state.plan = plan
        self.state.status = "planning"
        self._save_state("set_plan")
        print(f"Plan set with {plan['total_tasks']} tasks")
    
    def start_execution(self) -> None:
        """Start execution phase."""
        self.state.status = "executing"
        self.state.current_task_index = 0
        self._save_state("start_execution")
        print("Starting execution...")
    
    def update_task_result(self, task_id: str, result: Any) -> None:
        """Update state with task result."""
        self.state.add_result(task_id, result)
        self.state.current_task_index += 1
        self._save_state("update_result")
        print(f"Task {task_id} completed")
    
    def update_task_error(self, task_id: str, error: str) -> None:
        """Update state with task error."""
        self.state.add_error(task_id, error)
        self._save_state("update_error")
        print(f"Task {task_id} failed: {error}")
    
    def start_review(self) -> None:
        """Start review phase."""
        self.state.status = "reviewing"
        self._save_state("start_review")
        print("Starting review...")
    
    def complete(self, final_response: str) -> None:
        """Complete the agent workflow."""
        self.state.status = "completed"
        self.state.final_response = final_response
        self.state.end_time = datetime.now().isoformat()
        self._save_state("complete")
        print("Workflow completed")
    
    def fail(self, error: str) -> None:
        """Fail the agent workflow."""
        self.state.status = "failed"
        self.state.errors["workflow"] = error
        self.state.end_time = datetime.now().isoformat()
        self._save_state("fail")
        print(f"Workflow failed: {error}")
    
    def _save_state(self, transition: str) -> None:
        """Save current state to history."""
        self.state_history.append({
            "transition": transition,
            "timestamp": datetime.now().isoformat(),
            "state": self.state.to_dict()
        })
    
    def get_state(self) -> AgentState:
        """Get current state."""
        return self.state
    
    def get_history(self) -> List[Dict[str, Any]]:
        """Get state history."""
        return self.state_history
    
    def get_summary(self) -> Dict[str, Any]:
        """Get state summary."""
        return {
            "query": self.state.query,
            "status": self.state.status,
            "tasks_completed": len(self.state.results),
            "tasks_failed": len(self.state.errors),
            "total_tasks": self.state.plan.get('total_tasks', 0) if self.state.plan else 0,
            "start_time": self.state.start_time,
            "end_time": self.state.end_time,
            "has_final_response": bool(self.state.final_response)
        }
    
    def reset(self) -> None:
        """Reset state manager."""
        self.state = AgentState()
        self.state_history = []
        print("State manager reset")

def create_state_manager() -> StateManager:
    """Factory function to create a state manager."""
    return StateManager()

# Test the state manager
print("Testing State Manager...")
print("=" * 60)

# Import the TaskPlanner from CELL 2
# Since we're in the same notebook, we can use the planner instance

# Create state manager
state_manager = create_state_manager()

# Test with a sample plan
test_query = "What is Python programming and calculate 100/4"

print(f"\nTest Query: {test_query}")
print("-" * 40)

# Initialize state
state_manager.initialize(test_query)

# Create a plan using the planner from CELL 2
plan = planner.create_plan(test_query)

# Set the plan
state_manager.set_plan(plan)

# Start execution
state_manager.start_execution()

# Simulate task execution
tasks = plan.get('task_objects', [])
for i, task in enumerate(tasks):
    print(f"\nExecuting task {i+1}: {task.description}")
    task.status = "running"
    
    # Simulate success for first tasks, failure for last
    if i < len(tasks) - 1:
        result = f"Result for {task.task_id}: Success!"
        task.mark_completed(result)
        state_manager.update_task_result(task.task_id, result)
    else:
        error = f"Error in {task.task_id}"
        task.mark_failed(error)
        state_manager.update_task_error(task.task_id, error)

# Complete the workflow
state_manager.start_review()
state_manager.complete("Workflow completed with some errors")

print("\nState Summary:")
summary = state_manager.get_summary()
for key, value in summary.items():
    print(f"  - {key}: {value}")

print("\nState History:")
history = state_manager.get_history()
for entry in history[-3:]:
    print(f"  - {entry['transition']} at {entry['timestamp'][:19]}")

print("\n" + "=" * 60)
print("State Manager test completed.")

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


LangGraph imported successfully.
Testing State Manager...

Test Query: What is Python programming and calculate 100/4
----------------------------------------
State initialized for query: What is Python programming and calculate 100/4

Planning for query: What is Python programming and calculate 100/4
----------------------------------------
Created plan with 3 tasks:
  - task_013: Research: python programming and calculate [wikipedia_search]
  - task_014: Calculate: 100/4 [calculator]
  - task_015: Calculate: 100/4 [calculator]
Plan set with 3 tasks
Starting execution...

Executing task 1: Research: python programming and calculate
Task task_013 completed

Executing task 2: Calculate: 100/4
Task task_014 completed

Executing task 3: Calculate: 100/4
Task task_015 failed: Error in task_015
Starting review...
Workflow completed

State Summary:
  - query: What is Python programming and calculate 100/4
  - status: completed
  - tasks_completed: 2
  - tasks_failed: 1
  - total_tasks: 3
  -

In [6]:
# CELL 5: Complete Agent with Planning and Memory (Self-Contained)
# This cell integrates everything into a complete agent with planning and memory

import time
import json
import re
import math
from datetime import datetime
from typing import Dict, Any, Optional, List, Tuple

# Import required libraries
import requests
import wikipediaapi

print("Building Complete Planning Memory Agent...")
print("=" * 60)

# ============================================================================
# PART 1: Recreate Tools (from Day 1)
# ============================================================================

class CalculatorTool:
    """Calculator tool for mathematical operations."""
    name = "calculator"
    description = "Perform mathematical calculations"
    
    def __init__(self):
        self._cache = {}
    
    def _run(self, expression: str) -> str:
        try:
            expression = expression.strip()
            if not expression:
                return "Error: No expression provided"
            
            # Safe evaluation
            safe_context = {
                '__builtins__': {},
                'math': math,
                'sqrt': math.sqrt,
                'sin': math.sin,
                'cos': math.cos,
                'tan': math.tan,
                'log': math.log,
                'log10': math.log10,
                'abs': abs,
                'ceil': math.ceil,
                'floor': math.floor,
                'round': round,
                'pi': math.pi,
                'e': math.e
            }
            
            result = eval(expression, safe_context)
            return f"Result: {float(result)}"
        except Exception as e:
            return f"Error: {str(e)}"

class WikipediaTool:
    """Wikipedia search tool."""
    name = "wikipedia_search"
    description = "Search Wikipedia for information"
    
    def __init__(self):
        self._wiki = wikipediaapi.Wikipedia(
            language='en',
            user_agent='AgenticAI-Project/1.0'
        )
        self._cache = {}
    
    def _clean_text(self, text: str, max_length: int = 500) -> str:
        if not text:
            return "No content available."
        text = re.sub(r'\s+', ' ', text)
        if len(text) > max_length:
            text = text[:max_length] + "..."
        return text.strip()
    
    def _run(self, query: str, max_results: int = 2) -> str:
        try:
            page = self._wiki.page(query)
            if page.exists():
                return f"Wikipedia Article: {page.title}\n\n{self._clean_text(page.summary, 500)}\n\nURL: {page.fullurl}"
            
            # Search
            results = list(self._wiki.search(query))[:max_results]
            if not results:
                return f"No Wikipedia results found for: {query}"
            
            output = f"Wikipedia Search Results for '{query}':\n\n"
            for idx, title in enumerate(results, 1):
                page = self._wiki.page(title)
                if page.exists():
                    output += f"{idx}. {title}\n"
                    output += f"   {self._clean_text(page.summary, 200)}\n"
                    output += f"   URL: {page.fullurl}\n\n"
            return output
        except Exception as e:
            return f"Error searching Wikipedia: {str(e)}"

class WebSearchTool:
    """Web search tool using Wikipedia API."""
    name = "web_search"
    description = "Search the web for information"
    
    def __init__(self):
        self._cache = {}
    
    def _run(self, query: str, max_results: int = 2) -> str:
        try:
            url = "https://en.wikipedia.org/w/api.php"
            params = {
                "action": "query",
                "list": "search",
                "srsearch": query,
                "format": "json",
                "srlimit": max_results,
                "srprop": "snippet"
            }
            headers = {"User-Agent": "AgenticAI-Project/1.0"}
            response = requests.get(url, params=params, headers=headers, timeout=10)
            
            if response.status_code != 200:
                return f"No results found for: {query}"
            
            data = response.json()
            results = data.get("query", {}).get("search", [])
            
            if not results:
                return f"No results found for: {query}"
            
            output = f"Search Results for '{query}':\n\n"
            for idx, item in enumerate(results, 1):
                title = item.get("title", "")
                snippet = re.sub(r'<[^>]+>', '', item.get("snippet", ""))
                output += f"{idx}. {title}\n"
                output += f"   {snippet[:200]}...\n\n"
            return output
        except Exception as e:
            return f"Error performing search: {str(e)}"

# ============================================================================
# PART 2: Recreate Tool Registry
# ============================================================================

class ToolRegistry:
    """Registry for managing tools."""
    
    def __init__(self):
        self._tools = {}
    
    def register_tool(self, tool):
        self._tools[tool.name] = tool
        print(f"  Registered: {tool.name}")
    
    def get_tool(self, tool_name):
        return self._tools.get(tool_name)
    
    def list_tools(self):
        return list(self._tools.keys())
    
    def execute_tool(self, tool_name, **kwargs):
        tool = self.get_tool(tool_name)
        if not tool:
            return {"success": False, "error": f"Tool '{tool_name}' not found"}
        try:
            result = tool._run(**kwargs)
            return {"success": True, "result": result}
        except Exception as e:
            return {"success": False, "error": str(e)}

# ============================================================================
# PART 3: Recreate Task Planner (from CELL 2)
# ============================================================================

class Task:
    """Represents a single task."""
    
    def __init__(self, task_id: str, description: str, tool: str = None, 
                 priority: int = 1, dependencies: List[str] = None):
        self.task_id = task_id
        self.description = description
        self.tool = tool
        self.priority = priority
        self.dependencies = dependencies or []
        self.status = "pending"
        self.result = None
        self.error = None
        self.expression = None
        self.search_query = None
    
    def mark_completed(self, result):
        self.status = "completed"
        self.result = result
    
    def mark_failed(self, error):
        self.status = "failed"
        self.error = error

class TaskPlanner:
    """Task planner that decomposes queries into subtasks."""
    
    def __init__(self):
        self.task_counter = 0
    
    def _generate_task_id(self):
        self.task_counter += 1
        return f"task_{self.task_counter:03d}"
    
    def _extract_calculations(self, query: str) -> List[Dict]:
        calc_tasks = []
        patterns = [
            r'calculate\s+([\d+\-*/()\s.^]+)',
            r'what is\s+([\d+\-*/()\s.^]+)',
            r'compute\s+([\d+\-*/()\s.^]+)',
            r'(\d+\s*[\+\-\*/]\s*\d+)'
        ]
        
        for pattern in patterns:
            matches = re.findall(pattern, query.lower())
            for match in matches:
                expr = match.strip()
                if expr and re.search(r'[\d+\-*/]', expr):
                    expr = expr.replace('x', '*')
                    calc_tasks.append({
                        "task_id": self._generate_task_id(),
                        "description": f"Calculate: {expr}",
                        "tool": "calculator",
                        "priority": 2,
                        "dependencies": [],
                        "expression": expr
                    })
                    break  # Only take first calculation
        
        return calc_tasks
    
    def _extract_research(self, query: str) -> List[Dict]:
        research_tasks = []
        patterns = [
            r'what is\s+([a-zA-Z\s]+)',
            r'who is\s+([a-zA-Z\s]+)',
            r'define\s+([a-zA-Z\s]+)',
            r'about\s+([a-zA-Z\s]+)',
            r'research\s+([a-zA-Z\s]+)'
        ]
        
        for pattern in patterns:
            matches = re.findall(pattern, query.lower())
            for match in matches:
                topic = match.strip()
                if topic and len(topic) > 2 and not any(c in topic for c in ['+', '-', '*', '/']):
                    research_tasks.append({
                        "task_id": self._generate_task_id(),
                        "description": f"Research: {topic}",
                        "tool": "wikipedia_search",
                        "priority": 1,
                        "dependencies": [],
                        "search_query": topic
                    })
                    break  # Only take first research topic
        
        return research_tasks
    
    def create_plan(self, query: str) -> Dict[str, Any]:
        """Create a plan by decomposing the query."""
        print(f"\nPlanning for: {query}")
        print("-" * 40)
        
        all_tasks = []
        all_tasks.extend(self._extract_research(query))
        all_tasks.extend(self._extract_calculations(query))
        
        # Default search if no tasks found
        if not all_tasks:
            all_tasks.append({
                "task_id": self._generate_task_id(),
                "description": f"Search: {query[:50]}",
                "tool": "web_search",
                "priority": 1,
                "dependencies": [],
                "search_query": query
            })
        
        # Sort by priority
        all_tasks.sort(key=lambda x: x['priority'])
        
        # Create Task objects
        task_objects = []
        for data in all_tasks:
            task = Task(
                task_id=data['task_id'],
                description=data['description'],
                tool=data['tool'],
                priority=data['priority'],
                dependencies=data.get('dependencies', [])
            )
            if 'expression' in data:
                task.expression = data['expression']
            if 'search_query' in data:
                task.search_query = data['search_query']
            task_objects.append(task)
        
        plan = {
            "original_query": query,
            "total_tasks": len(task_objects),
            "task_objects": task_objects,
            "execution_order": [t.task_id for t in task_objects]
        }
        
        print(f"Created plan with {len(task_objects)} tasks:")
        for task in task_objects:
            print(f"  - {task.task_id}: {task.description} [{task.tool}]")
        
        return plan

# ============================================================================
# PART 4: Recreate Memory System (from CELL 3)
# ============================================================================

class MemoryEntry:
    """Represents a memory entry."""
    
    def __init__(self, content: str, memory_type: str = "short_term", importance: int = 1):
        self.content = content
        self.memory_type = memory_type
        self.importance = importance
        self.created_at = datetime.now().isoformat()
        self.access_count = 0
    
    def to_dict(self):
        return {
            "content": self.content,
            "memory_type": self.memory_type,
            "importance": self.importance,
            "created_at": self.created_at,
            "access_count": self.access_count
        }

class MemorySystem:
    """Memory system with short-term, long-term, and episodic memory."""
    
    def __init__(self, max_short_term: int = 10):
        self.short_term_memory = []
        self.long_term_memory = []
        self.episodic_memory = []
        self.max_short_term = max_short_term
    
    def store(self, content: str, memory_type: str = "short_term", importance: int = 1):
        entry = MemoryEntry(content, memory_type, importance)
        
        if memory_type == "short_term":
            self.short_term_memory.append(entry)
            if len(self.short_term_memory) > self.max_short_term:
                self.short_term_memory.pop(0)
        elif memory_type == "long_term":
            self.long_term_memory.append(entry)
        elif memory_type == "episodic":
            self.episodic_memory.append(entry)
        
        print(f"  Stored in {memory_type}: {content[:50]}...")
    
    def retrieve(self, query: str, limit: int = 3):
        results = []
        query_lower = query.lower()
        
        for pool in self.short_term_memory + self.long_term_memory + self.episodic_memory:
            if query_lower in pool.content.lower():
                pool.access_count += 1
                results.append(pool)
        
        results.sort(key=lambda x: (x.importance, x.access_count), reverse=True)
        return results[:limit]
    
    def summarize_memories(self):
        return {
            "short_term_count": len(self.short_term_memory),
            "long_term_count": len(self.long_term_memory),
            "episodic_count": len(self.episodic_memory),
            "total_memories": len(self.short_term_memory) + len(self.long_term_memory) + len(self.episodic_memory)
        }
    
    def get_all_memories(self):
        return {
            "short_term": [e.to_dict() for e in self.short_term_memory],
            "long_term": [e.to_dict() for e in self.long_term_memory],
            "episodic": [e.to_dict() for e in self.episodic_memory]
        }

# ============================================================================
# PART 5: Recreate State Manager (from CELL 4)
# ============================================================================

class AgentState:
    """Agent state management."""
    
    def __init__(self):
        self.query = ""
        self.plan = None
        self.results = {}
        self.errors = {}
        self.status = "initialized"
        self.start_time = None
        self.end_time = None
        self.final_response = ""

class StateManager:
    """Manages agent state."""
    
    def __init__(self):
        self.state = AgentState()
    
    def initialize(self, query: str):
        self.state.query = query
        self.state.status = "initialized"
        self.state.start_time = datetime.now().isoformat()
        print(f"State initialized for: {query[:50]}...")
    
    def set_plan(self, plan: Dict):
        self.state.plan = plan
        self.state.status = "planning"
    
    def start_execution(self):
        self.state.status = "executing"
        print("Starting execution...")
    
    def update_task_result(self, task_id: str, result: Any):
        self.state.results[task_id] = result
    
    def update_task_error(self, task_id: str, error: str):
        self.state.errors[task_id] = error
    
    def start_review(self):
        self.state.status = "reviewing"
        print("Reviewing results...")
    
    def complete(self, response: str):
        self.state.status = "completed"
        self.state.final_response = response
        self.state.end_time = datetime.now().isoformat()
    
    def get_summary(self):
        return {
            "query": self.state.query,
            "status": self.state.status,
            "tasks_completed": len(self.state.results),
            "tasks_failed": len(self.state.errors)
        }

# ============================================================================
# PART 6: Complete Agent
# ============================================================================

class PlanningMemoryAgent:
    """Complete agent with planning and memory."""
    
    def __init__(self, registry, planner, memory_system, state_manager):
        self.registry = registry
        self.planner = planner
        self.memory = memory_system
        self.state_manager = state_manager
        self.conversation_history = []
        
        print(f"\nAgent initialized with tools: {registry.list_tools()}")
    
    def _execute_task(self, task):
        tool = self.registry.get_tool(task.tool)
        if not tool:
            return False, f"Tool '{task.tool}' not found"
        
        kwargs = {}
        if task.tool == "calculator" and hasattr(task, 'expression'):
            kwargs['expression'] = task.expression
        elif hasattr(task, 'search_query'):
            kwargs['query'] = task.search_query
        else:
            # Extract from description
            desc = task.description
            if 'Calculate:' in desc or 'calculate' in desc.lower():
                expr = re.sub(r'(Calculate:|calculate|Compute:|compute)', '', desc).strip()
                kwargs['expression'] = expr
            else:
                kwargs['query'] = desc.replace('Research:', '').replace('Search:', '').strip()
        
        print(f"    Executing {task.tool}...")
        result = self.registry.execute_tool(task.tool, **kwargs)
        
        if result.get('success'):
            return True, result['result']
        else:
            return False, result.get('error', 'Unknown error')
    
    def process_query(self, query: str) -> str:
        print("\n" + "=" * 60)
        print(f"Processing: {query}")
        print("=" * 60)
        
        # Initialize
        self.state_manager.initialize(query)
        
        # Check memory
        print("\nChecking memory...")
        memories = self.memory.retrieve(query, limit=2)
        if memories:
            print(f"  Found {len(memories)} relevant memories")
        
        # Create plan
        print("\nCreating plan...")
        plan = self.planner.create_plan(query)
        self.state_manager.set_plan(plan)
        
        # Store query in memory
        self.memory.store(query, "short_term", 5)
        
        # Execute tasks
        print("\nExecuting tasks...")
        self.state_manager.start_execution()
        
        tasks = plan.get('task_objects', [])
        successful = []
        failed = []
        
        for idx, task in enumerate(tasks, 1):
            print(f"\nTask {idx}/{len(tasks)}: {task.description}")
            task.status = "running"
            
            success, result = self._execute_task(task)
            
            if success:
                task.mark_completed(result)
                self.state_manager.update_task_result(task.task_id, result)
                successful.append((task.description, result))
                self.memory.store(result[:100] if result else "Completed", "episodic", 3)
            else:
                task.mark_failed(result)
                self.state_manager.update_task_error(task.task_id, result)
                failed.append((task.description, result))
            
            time.sleep(0.2)
        
        # Generate response
        print("\nGenerating response...")
        self.state_manager.start_review()
        
        response = self._format_response(query, successful, failed)
        self.state_manager.complete(response)
        
        self.conversation_history.append({
            "query": query,
            "successful": len(successful),
            "failed": len(failed)
        })
        
        return response
    
    def _format_response(self, query: str, successful: List[Tuple], failed: List[Tuple]) -> str:
        lines = []
        lines.append(f"Query: {query}")
        lines.append("")
        lines.append("-" * 50)
        
        if successful:
            lines.append("RESULTS:")
            lines.append("")
            for desc, result in successful:
                lines.append(f"[{desc}]")
                if result and len(result) > 300:
                    result = result[:300] + "..."
                lines.append(f"{result}")
                lines.append("")
        
        if failed:
            lines.append("FAILED TASKS:")
            for desc, error in failed:
                lines.append(f"  - {desc}: {error}")
        
        lines.append("-" * 50)
        summary = self.memory.summarize_memories()
        lines.append(f"Memory: {summary['total_memories']} memories stored")
        
        return "\n".join(lines)

# ============================================================================
# PART 7: Build Everything
# ============================================================================

print("\nBuilding all components...")
print("-" * 40)

# Create registry and register tools
registry = ToolRegistry()
print("\nRegistering tools:")
registry.register_tool(WikipediaTool())
registry.register_tool(WebSearchTool())
registry.register_tool(CalculatorTool())

# Create other components
planner = TaskPlanner()
memory = MemorySystem()
state_manager = StateManager()

# Create agent
agent = PlanningMemoryAgent(registry, planner, memory, state_manager)

# ============================================================================
# PART 8: Test the Agent
# ============================================================================

print("\n" + "=" * 60)
print("TESTING THE COMPLETE AGENT")
print("=" * 60)

test_queries = [
    "What is artificial intelligence and calculate 100/4",
    "Research machine learning",
    "What is Python programming"
]

for query in test_queries:
    print("\n" + "=" * 60)
    response = agent.process_query(query)
    print("\nFINAL RESPONSE:")
    print(response)

print("\n" + "=" * 60)
print("Conversation Summary:")
for i, entry in enumerate(agent.conversation_history, 1):
    print(f"  Query {i}: {entry['query'][:40]}...")
    print(f"    Success: {entry['successful']}, Failed: {entry['failed']}")

# ============================================================================
# PART 9: Save Data
# ============================================================================

print("\n" + "=" * 60)
print("Saving Day 2 Data...")

agent_data = {
    "conversation_history": agent.conversation_history,
    "memory_summary": memory.summarize_memories(),
    "state_summary": state_manager.get_summary(),
    "timestamp": datetime.now().isoformat()
}

with open("day2_agent_data.json", "w") as f:
    json.dump(agent_data, f, indent=2)
print("  Saved day2_agent_data.json")

memory_data = memory.get_all_memories()
with open("day2_memory_data.json", "w") as f:
    json.dump(memory_data, f, indent=2)
print("  Saved day2_memory_data.json")

print("\n" + "=" * 60)
print("Day 2 Complete!")

Building Complete Planning Memory Agent...

Building all components...
----------------------------------------

Registering tools:
  Registered: wikipedia_search
  Registered: web_search
  Registered: calculator

Agent initialized with tools: ['wikipedia_search', 'web_search', 'calculator']

TESTING THE COMPLETE AGENT


Processing: What is artificial intelligence and calculate 100/4
State initialized for: What is artificial intelligence and calculate 100/...

Checking memory...

Creating plan...

Planning for: What is artificial intelligence and calculate 100/4
----------------------------------------
Created plan with 3 tasks:
  - task_001: Research: artificial intelligence and calculate [wikipedia_search]
  - task_002: Calculate: 100/4 [calculator]
  - task_003: Calculate: 100/4 [calculator]
  Stored in short_term: What is artificial intelligence and calculate 100/...

Executing tasks...
Starting execution...

Task 1/3: Research: artificial intelligence and calculate
    Executing w

In [7]:
# CELL 6: Enhanced Planning with Dependencies and Parallel Execution
# This cell adds advanced planning capabilities

from typing import Dict, Any, Optional, List, Set
import networkx as nx
from collections import deque

class AdvancedPlanner:
    """
    Advanced planner with dependency resolution and parallel execution support.
    """
    
    def __init__(self, planner):
        self.planner = planner
        self.execution_graph = None
    
    def create_dependency_graph(self, plan: Dict[str, Any]) -> nx.DiGraph:
        """
        Create a dependency graph from a plan.
        
        Args:
            plan: The plan dictionary
            
        Returns:
            NetworkX directed graph
        """
        G = nx.DiGraph()
        tasks = plan.get('task_objects', [])
        
        for task in tasks:
            G.add_node(task.task_id, task=task)
            
        for task in tasks:
            for dep_id in task.dependencies:
                if dep_id in G.nodes:
                    G.add_edge(dep_id, task.task_id)
        
        self.execution_graph = G
        return G
    
    def get_parallel_groups(self, plan: Dict[str, Any]) -> List[List[str]]:
        """
        Identify groups of tasks that can be executed in parallel.
        
        Args:
            plan: The plan dictionary
            
        Returns:
            List of groups, each group is a list of task IDs
        """
        G = self.create_dependency_graph(plan)
        
        # Get topological order
        try:
            topo_order = list(nx.topological_sort(G))
        except nx.NetworkXUnfeasible:
            return []
        
        # Group tasks by level
        levels = {}
        for task_id in topo_order:
            predecessors = list(G.predecessors(task_id))
            if not predecessors:
                levels[task_id] = 0
            else:
                levels[task_id] = max(levels[p] for p in predecessors) + 1
        
        # Group by level
        groups = {}
        for task_id, level in levels.items():
            if level not in groups:
                groups[level] = []
            groups[level].append(task_id)
        
        return [groups[level] for level in sorted(groups.keys())]
    
    def estimate_execution_time(self, plan: Dict[str, Any]) -> float:
        """
        Estimate total execution time based on task dependencies.
        
        Args:
            plan: The plan dictionary
            
        Returns:
            Estimated time in seconds
        """
        # Average execution times per tool
        tool_times = {
            'wikipedia_search': 0.8,
            'web_search': 1.2,
            'calculator': 0.2,
            'default': 0.5
        }
        
        G = self.create_dependency_graph(plan)
        times = {}
        
        for task_id in nx.topological_sort(G):
            task = G.nodes[task_id]['task']
            tool = task.tool or 'default'
            task_time = tool_times.get(tool, tool_times['default'])
            
            predecessors = list(G.predecessors(task_id))
            if predecessors:
                max_pred_time = max(times[p] for p in predecessors)
                times[task_id] = max_pred_time + task_time
            else:
                times[task_id] = task_time
        
        if times:
            return max(times.values())
        return 0.0
    
    def get_parallel_plan(self, plan: Dict[str, Any]) -> Dict[str, Any]:
        """
        Generate a plan with parallel execution groups.
        
        Args:
            plan: The original plan
            
        Returns:
            Enhanced plan with parallel groups
        """
        groups = self.get_parallel_groups(plan)
        
        enhanced_plan = plan.copy()
        enhanced_plan['parallel_groups'] = groups
        enhanced_plan['estimated_time'] = self.estimate_execution_time(plan)
        
        print(f"\nParallel Execution Plan:")
        print(f"  Total tasks: {plan['total_tasks']}")
        print(f"  Parallel groups: {len(groups)}")
        print(f"  Estimated time: {enhanced_plan['estimated_time']:.2f}s")
        
        for i, group in enumerate(groups, 1):
            print(f"  Group {i}: {group}")
        
        return enhanced_plan

def create_advanced_planner(planner) -> AdvancedPlanner:
    """Factory function to create an advanced planner."""
    return AdvancedPlanner(planner)

# Test the advanced planner
print("Testing Advanced Planner...")
print("=" * 60)

# Create a sample plan with dependencies
sample_plan = planner.create_plan("Research AI and calculate 100/4 and find latest news")

# Create advanced planner
advanced_planner = create_advanced_planner(planner)

# Get parallel groups
parallel_plan = advanced_planner.get_parallel_plan(sample_plan)

print("\n" + "=" * 60)
print("Advanced Planner test completed.")

Testing Advanced Planner...

Planning for: Research AI and calculate 100/4 and find latest news
----------------------------------------
Created plan with 3 tasks:
  - task_006: Research: ai and calculate [wikipedia_search]
  - task_007: Calculate: 100/4 [calculator]
  - task_008: Calculate: 100/4 [calculator]

Parallel Execution Plan:
  Total tasks: 3
  Parallel groups: 1
  Estimated time: 0.80s
  Group 1: ['task_006', 'task_007', 'task_008']

Advanced Planner test completed.


In [8]:
# CELL 7: Episodic Memory with Importance Scoring
# This cell adds episodic memory with importance scoring and consolidation

import numpy as np
from collections import defaultdict

class EpisodicMemory:
    """
    Enhanced episodic memory with importance scoring and consolidation.
    """
    
    def __init__(self, memory_system):
        self.memory = memory_system
        self.episode_counter = 0
        self.importance_threshold = 5
        
    def record_episode(self, query: str, actions: List[Dict], outcome: str, 
                       success: bool, importance_score: int = None):
        """
        Record an episode (complete interaction).
        
        Args:
            query: The user query
            actions: List of actions taken
            outcome: Final outcome
            success: Whether the episode was successful
            importance_score: Importance score (1-10)
        """
        self.episode_counter += 1
        
        if importance_score is None:
            importance_score = self._calculate_importance(actions, success)
        
        episode = {
            "episode_id": self.episode_counter,
            "query": query,
            "actions": actions,
            "outcome": outcome,
            "success": success,
            "importance": importance_score,
            "timestamp": datetime.now().isoformat()
        }
        
        # Store in memory
        content = f"Episode {self.episode_counter}: {query} -> {outcome[:100]}"
        self.memory.store(content, "episodic", importance_score)
        
        # Store full episode details
        self.memory.store(json.dumps(episode), "long_term", importance_score)
        
        print(f"  Recorded episode {self.episode_counter} (importance: {importance_score})")
        
        return episode
    
    def _calculate_importance(self, actions: List[Dict], success: bool) -> int:
        """
        Calculate importance score based on actions and outcome.
        
        Args:
            actions: List of actions
            success: Whether successful
            
        Returns:
            Importance score (1-10)
        """
        score = 5  # Base score
        
        # More actions = higher importance
        if len(actions) > 3:
            score += 2
        
        # Success adds importance
        if success:
            score += 1
        else:
            score += 2  # Learn from failures too
        
        # Types of actions
        tools_used = set()
        for action in actions:
            if 'tool' in action:
                tools_used.add(action['tool'])
        
        if 'wikipedia_search' in tools_used:
            score += 1
        if 'calculator' in tools_used:
            score += 1
        
        # Cap at 10
        return min(score, 10)
    
    def get_important_episodes(self, min_importance: int = 7) -> List[Dict]:
        """
        Get episodes with importance above threshold.
        
        Args:
            min_importance: Minimum importance threshold
            
        Returns:
            List of important episodes
        """
        important = []
        for entry in self.memory.long_term_memory:
            try:
                data = json.loads(entry.content)
                if data.get('importance', 0) >= min_importance:
                    important.append(data)
            except:
                continue
        
        return sorted(important, key=lambda x: x.get('importance', 0), reverse=True)
    
    def consolidate_memories(self, min_importance: int = 8):
        """
        Consolidate high-importance memories into long-term memory.
        
        Args:
            min_importance: Minimum importance for consolidation
        """
        consolidated = []
        
        # Check short-term memories
        for entry in self.memory.short_term_memory[:]:
            if entry.importance >= min_importance:
                self.memory.store(entry.content, "long_term", entry.importance)
                consolidated.append(entry.content)
                self.memory.short_term_memory.remove(entry)
        
        if consolidated:
            print(f"Consolidated {len(consolidated)} memories to long-term")
        
        return consolidated

def create_episodic_memory(memory_system) -> EpisodicMemory:
    """Factory function to create episodic memory."""
    return EpisodicMemory(memory_system)

# Test episodic memory
print("Testing Episodic Memory...")
print("=" * 60)

episodic = create_episodic_memory(memory)

# Record some episodes
episodic.record_episode(
    query="What is AI?",
    actions=[{"tool": "wikipedia_search", "query": "Artificial Intelligence"}],
    outcome="Found AI definition",
    success=True
)

episodic.record_episode(
    query="Calculate 50*30",
    actions=[{"tool": "calculator", "expression": "50*30"}],
    outcome="1500",
    success=True
)

# Get important episodes
important = episodic.get_important_episodes()
print(f"\nImportant episodes: {len(important)}")

print("\n" + "=" * 60)
print("Episodic Memory test completed.")

Testing Episodic Memory...
  Stored in episodic: Episode 1: What is AI? -> Found AI definition...
  Stored in long_term: {"episode_id": 1, "query": "What is AI?", "actions...
  Recorded episode 1 (importance: 7)
  Stored in episodic: Episode 2: Calculate 50*30 -> 1500...
  Stored in long_term: {"episode_id": 2, "query": "Calculate 50*30", "act...
  Recorded episode 2 (importance: 7)

Important episodes: 2

Episodic Memory test completed.


In [9]:
# CELL 8: Human-in-the-Loop and Review Node
# This cell adds human-in-the-loop capability and review functionality

class HumanInTheLoop:
    """
    Human-in-the-loop capability for agent workflows.
    """
    
    def __init__(self, state_manager):
        self.state_manager = state_manager
        self.approval_required = True
        self.approval_history = []
    
    def request_approval(self, task: Task, result: Any) -> bool:
        """
        Request human approval for a task result.
        
        Args:
            task: The task being reviewed
            result: The result to approve
            
        Returns:
            True if approved, False otherwise
        """
        print(f"\nHUMAN APPROVAL REQUIRED:")
        print(f"  Task: {task.description}")
        print(f"  Tool: {task.tool}")
        print(f"  Result: {result[:200] if result else 'No result'}...")
        print("-" * 40)
        
        # In a real system, this would be a UI prompt
        # For demo, auto-approve after 3 seconds
        print("  Auto-approving in 3 seconds...")
        time.sleep(3)
        
        approved = True
        self.approval_history.append({
            "task_id": task.task_id,
            "description": task.description,
            "approved": approved,
            "timestamp": datetime.now().isoformat()
        })
        
        print(f"  Status: {'APPROVED' if approved else 'REJECTED'}")
        return approved
    
    def review_plan(self, plan: Dict[str, Any]) -> bool:
        """
        Review and approve a plan before execution.
        
        Args:
            plan: The plan to review
            
        Returns:
            True if approved, False otherwise
        """
        print(f"\nPLAN REVIEW:")
        print(f"  Query: {plan.get('original_query', 'Unknown')}")
        print(f"  Total tasks: {plan.get('total_tasks', 0)}")
        print("-" * 40)
        
        for i, task in enumerate(plan.get('task_objects', []), 1):
            deps = f" (depends: {task.dependencies})" if task.dependencies else ""
            print(f"  {i}. {task.description} [{task.tool}]{deps}")
        
        print("-" * 40)
        print("  Auto-approving plan in 3 seconds...")
        time.sleep(3)
        
        approved = True
        print(f"  Plan: {'APPROVED' if approved else 'REJECTED'}")
        return approved
    
    def can_execute_without_approval(self, task: Task) -> bool:
        """
        Check if a task can be executed without human approval.
        
        Args:
            task: The task to check
            
        Returns:
            True if no approval needed
        """
        # Low-priority or safe tasks
        if task.priority > 3:  # Lower priority = higher number
            return True
        
        # Calculator operations are safe
        if task.tool == "calculator":
            return True
        
        return False

def create_human_in_the_loop(state_manager) -> HumanInTheLoop:
    """Factory function to create human-in-the-loop."""
    return HumanInTheLoop(state_manager)

# Test HITL
print("Testing Human-in-the-Loop...")
print("=" * 60)

hitl = create_human_in_the_loop(state_manager)

# Create a sample task
sample_task = Task(
    task_id="test_001",
    description="Research AI safety",
    tool="wikipedia_search",
    priority=1
)

# Test approval
result = "Artificial Intelligence safety research results..."
approved = hitl.request_approval(sample_task, result)
print(f"\nFinal approval: {approved}")

# Test plan review
plan = planner.create_plan("Research AI and calculate 100/4")
hitl.review_plan(plan)

print("\n" + "=" * 60)
print("Human-in-the-Loop test completed.")

Testing Human-in-the-Loop...

HUMAN APPROVAL REQUIRED:
  Task: Research AI safety
  Tool: wikipedia_search
  Result: Artificial Intelligence safety research results......
----------------------------------------
  Auto-approving in 3 seconds...
  Status: APPROVED

Final approval: True

Planning for: Research AI and calculate 100/4
----------------------------------------
Created plan with 3 tasks:
  - task_009: Research: ai and calculate [wikipedia_search]
  - task_010: Calculate: 100/4 [calculator]
  - task_011: Calculate: 100/4 [calculator]

PLAN REVIEW:
  Query: Research AI and calculate 100/4
  Total tasks: 3
----------------------------------------
  1. Research: ai and calculate [wikipedia_search]
  2. Calculate: 100/4 [calculator]
  3. Calculate: 100/4 [calculator]
----------------------------------------
  Auto-approving plan in 3 seconds...
  Plan: APPROVED

Human-in-the-Loop test completed.


In [10]:
# CELL 9: Performance Metrics and Monitoring
# This cell adds performance tracking and monitoring capabilities

class PerformanceMonitor:
    """
    Performance monitoring for the agent.
    """
    
    def __init__(self):
        self.metrics = {
            "queries": [],
            "tool_calls": [],
            "execution_times": [],
            "success_count": 0,
            "failure_count": 0
        }
        self.session_start = datetime.now()
    
    def record_query(self, query: str, response: str, execution_time: float, 
                     success: bool, tasks: int):
        """
        Record a query execution.
        
        Args:
            query: The query
            response: The response
            execution_time: Time taken
            success: Whether successful
            tasks: Number of tasks
        """
        self.metrics["queries"].append({
            "query": query,
            "response_preview": response[:200],
            "execution_time": execution_time,
            "success": success,
            "tasks": tasks,
            "timestamp": datetime.now().isoformat()
        })
        
        if success:
            self.metrics["success_count"] += 1
        else:
            self.metrics["failure_count"] += 1
        
        self.metrics["execution_times"].append(execution_time)
    
    def record_tool_call(self, tool_name: str, execution_time: float, 
                         success: bool, args: Dict = None):
        """
        Record a tool call.
        
        Args:
            tool_name: Name of the tool
            execution_time: Time taken
            success: Whether successful
            args: Arguments passed
        """
        self.metrics["tool_calls"].append({
            "tool_name": tool_name,
            "execution_time": execution_time,
            "success": success,
            "args": args,
            "timestamp": datetime.now().isoformat()
        })
    
    def get_summary(self) -> Dict[str, Any]:
        """
        Get performance summary.
        
        Returns:
            Dictionary with performance metrics
        """
        total_queries = len(self.metrics["queries"])
        total_tool_calls = len(self.metrics["tool_calls"])
        
        if self.metrics["execution_times"]:
            avg_time = sum(self.metrics["execution_times"]) / len(self.metrics["execution_times"])
            min_time = min(self.metrics["execution_times"])
            max_time = max(self.metrics["execution_times"])
        else:
            avg_time = min_time = max_time = 0
        
        # Tool usage stats
        tool_stats = defaultdict(int)
        tool_success = defaultdict(int)
        for call in self.metrics["tool_calls"]:
            tool_stats[call["tool_name"]] += 1
            if call["success"]:
                tool_success[call["tool_name"]] += 1
        
        return {
            "total_queries": total_queries,
            "successful_queries": self.metrics["success_count"],
            "failure_queries": self.metrics["failure_count"],
            "success_rate": self.metrics["success_count"] / total_queries if total_queries > 0 else 0,
            "total_tool_calls": total_tool_calls,
            "avg_execution_time": avg_time,
            "min_execution_time": min_time,
            "max_execution_time": max_time,
            "tool_usage": dict(tool_stats),
            "tool_success_rate": {
                tool: tool_success[tool] / count if count > 0 else 0
                for tool, count in tool_stats.items()
            },
            "session_duration": (datetime.now() - self.session_start).total_seconds(),
            "timestamp": datetime.now().isoformat()
        }
    
    def generate_report(self) -> str:
        """
        Generate a formatted performance report.
        
        Returns:
            Formatted report string
        """
        summary = self.get_summary()
        
        report = []
        report.append("=" * 60)
        report.append("PERFORMANCE MONITORING REPORT")
        report.append("=" * 60)
        report.append(f"Generated: {datetime.now().isoformat()}")
        report.append(f"Session Duration: {summary['session_duration']:.1f}s")
        report.append("")
        
        report.append("QUERY STATISTICS:")
        report.append("-" * 40)
        report.append(f"  Total Queries: {summary['total_queries']}")
        report.append(f"  Successful: {summary['successful_queries']}")
        report.append(f"  Failed: {summary['failure_queries']}")
        report.append(f"  Success Rate: {summary['success_rate']*100:.1f}%")
        report.append(f"  Avg Execution Time: {summary['avg_execution_time']:.3f}s")
        report.append("")
        
        report.append("TOOL STATISTICS:")
        report.append("-" * 40)
        for tool, count in summary.get('tool_usage', {}).items():
            success_rate = summary['tool_success_rate'].get(tool, 0) * 100
            report.append(f"  {tool}: {count} calls, {success_rate:.0f}% success")
        report.append("")
        
        report.append("=" * 60)
        
        return "\n".join(report)

def create_performance_monitor() -> PerformanceMonitor:
    """Factory function to create a performance monitor."""
    return PerformanceMonitor()

# Test performance monitor
print("Testing Performance Monitor...")
print("=" * 60)

monitor = create_performance_monitor()

# Record some queries
monitor.record_query("What is AI", "AI stands for Artificial Intelligence...", 1.2, True, 2)
monitor.record_query("Calculate 50+30", "Result: 80", 0.3, True, 1)
monitor.record_query("Find news", "No results found", 0.5, False, 1)

# Record tool calls
monitor.record_tool_call("wikipedia_search", 0.8, True, {"query": "AI"})
monitor.record_tool_call("calculator", 0.2, True, {"expression": "50+30"})
monitor.record_tool_call("web_search", 0.5, False, {"query": "news"})

print("\n" + monitor.generate_report())

print("\n" + "=" * 60)
print("Performance Monitor test completed.")

Testing Performance Monitor...

PERFORMANCE MONITORING REPORT
Generated: 2026-09-01T06:56:50.800848
Session Duration: 0.0s

QUERY STATISTICS:
----------------------------------------
  Total Queries: 3
  Successful: 2
  Failed: 1
  Success Rate: 66.7%
  Avg Execution Time: 0.667s

TOOL STATISTICS:
----------------------------------------
  wikipedia_search: 1 calls, 100% success
  calculator: 1 calls, 100% success
  web_search: 1 calls, 0% success


Performance Monitor test completed.


In [12]:
# CELL 10: Interactive Demo with Gradio UI (Fixed)
# This cell creates a web interface with proper chat format

import gradio as gr
import time
from datetime import datetime

print("Building Interactive Agent Demo...")
print("=" * 60)

class AgentInterface:
    """
    Wrapper for the planning memory agent to work with Gradio.
    """
    
    def __init__(self, agent, monitor, hitl):
        self.agent = agent
        self.monitor = monitor
        self.hitl = hitl
        self.conversation = []
    
    def process_message(self, message, history):
        """
        Process a message and return response.
        
        Args:
            message: User message
            history: Chat history as list of tuples
            
        Returns:
            Response and updated history
        """
        if not message or message.strip() == "":
            return history + [(message, "Please enter a query.")]
        
        start_time = time.time()
        
        print(f"\n[{datetime.now().strftime('%H:%M:%S')}] User: {message}")
        
        try:
            response = self.agent.process_query(message)
            
            # Record metrics
            execution_time = time.time() - start_time
            self.monitor.record_query(
                query=message,
                response=response,
                execution_time=execution_time,
                success=True,
                tasks=len(self.agent.state_manager.state.results) if hasattr(self.agent, 'state_manager') else 0
            )
            
            self.conversation.append({
                "query": message,
                "response": response,
                "timestamp": datetime.now().isoformat()
            })
            
            print(f"[{datetime.now().strftime('%H:%M:%S')}] Response generated in {execution_time:.2f}s")
            
            # Return as list of tuples for Chatbot
            history.append((message, response))
            return history
            
        except Exception as e:
            error_msg = f"Error processing query: {str(e)}"
            history.append((message, error_msg))
            self.monitor.record_query(
                query=message,
                response=error_msg,
                execution_time=time.time() - start_time,
                success=False,
                tasks=0
            )
            return history
    
    def get_agent_info(self):
        """Get information about the agent."""
        tools = self.agent.registry.list_tools()
        memory_summary = self.agent.memory.summarize_memories()
        state_summary = self.agent.state_manager.get_summary()
        performance = self.monitor.get_summary()
        
        info = f"""
### Agent Information

**Tools Available:**
{chr(10).join(['- ' + tool for tool in tools])}

**Memory Status:**
- Short-term: {memory_summary['short_term_count']}
- Long-term: {memory_summary['long_term_count']}
- Episodic: {memory_summary['episodic_count']}
- Total: {memory_summary['total_memories']}

**Performance:**
- Total Queries: {performance['total_queries']}
- Success Rate: {performance['success_rate']*100:.1f}%
- Avg Execution Time: {performance['avg_execution_time']:.3f}s

**State:**
- Status: {state_summary['status']}
- Tasks Completed: {state_summary['tasks_completed']}
- Tasks Failed: {state_summary['tasks_failed']}
"""
        return info
    
    def clear_conversation(self):
        """Clear the conversation history."""
        self.conversation = []
        self.agent.memory.clear_short_term()
        self.agent.state_manager.reset()
        return [], "Conversation and short-term memory cleared."

# Create the interface
interface = AgentInterface(agent, monitor, hitl)

# Gradio UI
with gr.Blocks(title="Day 2: Planning and Memory Agent", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # Day 2: Planning and Memory Agent
    ## Interactive Demo with Task Planning and Memory
    
    This agent can:
    - **Plan** complex queries by breaking them into subtasks
    - **Remember** previous interactions (short-term, long-term, episodic)
    - **Execute** tools in sequence with dependency management
    - **Review** results before final response
    
    ### Sample Queries to Try:
    - *"What is artificial intelligence and calculate 100/4"*
    - *"Research machine learning and compute 50*30"*
    - *"Find information about Python programming"*
    - *"What is quantum computing? Also what is 2+2"*
    """)
    
    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(
                height=500,
                label="Conversation",
                bubble_full_width=False
            )
            
            with gr.Row():
                msg = gr.Textbox(
                    placeholder="Enter your query here...",
                    label="Query",
                    container=False,
                    scale=8
                )
                submit_btn = gr.Button("Send", variant="primary", scale=1)
            
            with gr.Row():
                clear_btn = gr.Button("Clear Conversation", size="sm")
                info_btn = gr.Button("Show Agent Info", size="sm")
        
        with gr.Column(scale=1):
            info_box = gr.Markdown("### Agent Status\n\nReady to process queries...")
    
    # Define the chat response function
    def respond(message, history):
        if not message or message.strip() == "":
            return history, ""
        new_history = interface.process_message(message, history)
        return new_history, ""
    
    # Update info function
    def update_info():
        return interface.get_agent_info()
    
    # Clear function
    def clear_chat():
        return [], "Conversation cleared."
    
    # Event handlers
    submit_btn.click(
        respond,
        inputs=[msg, chatbot],
        outputs=[chatbot, msg]
    )
    
    msg.submit(
        respond,
        inputs=[msg, chatbot],
        outputs=[chatbot, msg]
    )
    
    clear_btn.click(
        clear_chat,
        outputs=[chatbot, info_box]
    )
    
    info_btn.click(
        update_info,
        outputs=[info_box]
    )

print("\n" + "=" * 60)
print("Launching Gradio Interface...")
print("=" * 60)

# Launch the interface
demo.launch(share=True)

print("\n" + "=" * 60)
print("Day 2 Complete!")
print("All modules built and tested:")
print("  1. Task Planner (CELL 2)")
print("  2. Memory System (CELL 3)")
print("  3. State Manager (CELL 4)")
print("  4. Complete Agent (CELL 5)")
print("  5. Advanced Planner (CELL 6)")
print("  6. Episodic Memory (CELL 7)")
print("  7. Human-in-the-Loop (CELL 8)")
print("  8. Performance Monitor (CELL 9)")
print("  9. Interactive Demo (CELL 10)")

Building Interactive Agent Demo...

Launching Gradio Interface...
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://325aea1c7533ee9c52.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



Day 2 Complete!
All modules built and tested:
  1. Task Planner (CELL 2)
  2. Memory System (CELL 3)
  3. State Manager (CELL 4)
  4. Complete Agent (CELL 5)
  5. Advanced Planner (CELL 6)
  6. Episodic Memory (CELL 7)
  7. Human-in-the-Loop (CELL 8)
  8. Performance Monitor (CELL 9)
  9. Interactive Demo (CELL 10)

[07:13:19] User: Research machine learning

Processing: Research machine learning
State initialized for: Research machine learning...

Checking memory...
  Found 1 relevant memories

Creating plan...

Planning for: Research machine learning
----------------------------------------
Created plan with 1 tasks:
  - task_015: Research: machine learning [wikipedia_search]
  Stored in short_term: Research machine learning...

Executing tasks...
Starting execution...

Task 1/1: Research: machine learning
    Executing wikipedia_search...
  Stored in episodic: Wikipedia Article: Machine learning

Machine learn...

Generating response...
Reviewing results...
[07:13:21] Response gene

In [16]:
# CELL 11: Simple Wikipedia Search with Better Handling
# This cell provides a simpler, more reliable Wikipedia search

import requests
import urllib.parse
import re

def simple_wikipedia_search(query: str) -> str:
    """
    Simple Wikipedia search using the API with better error handling.
    """
    try:
        url = "https://en.wikipedia.org/w/api.php"
        params = {
            "action": "query",
            "list": "search",
            "srsearch": query,
            "format": "json",
            "srlimit": 2,
            "srprop": "snippet"
        }
        
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, params=params, headers=headers, timeout=5)
        
        if response.status_code != 200:
            return ""
        
        data = response.json()
        results = data.get("query", {}).get("search", [])
        
        if not results:
            return ""
        
        output = []
        for item in results[:2]:
            title = item.get("title", "")
            snippet = re.sub(r'<[^>]+>', '', item.get("snippet", ""))
            output.append(f"{title}: {snippet[:150]}...")
        
        return "\n".join(output) if output else ""
        
    except:
        return ""

# Update the Wikipedia tool to use this simple function
class SimpleWikipediaTool:
    name = "wikipedia_search"
    description = "Search Wikipedia"
    
    def _run(self, query: str, max_results: int = 2) -> str:
        result = simple_wikipedia_search(query)
        if result:
            return f"Wikipedia Results:\n\n{result}"
        return f"Note: No Wikipedia article found for '{query}'. Try a simpler query."

# Update registry
registry._tools['wikipedia_search'] = SimpleWikipediaTool()

print("Wikipedia tool updated to simple version.")
print("Testing with: rag model")
result = registry.execute_tool("wikipedia_search", query="rag model")
print(result.get('result', ''))

Wikipedia tool updated to simple version.
Testing with: rag model
Wikipedia Results:

Retrieval-augmented generation: generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources. With...
Large language model: (RAG), fine-tuning, and other methods. The matter of LLM&#039;s exhibiting intelligence or understanding has two main aspects—the first is how to mode...


In [17]:
# CELL 12: Updated Agent with Better Response for Missing Data
# This cell updates the agent to handle missing data gracefully

class UpdatedPlanningAgent:
    """
    Updated agent with better handling for missing data.
    """
    
    def __init__(self, original_agent):
        self.original_agent = original_agent
        self.registry = original_agent.registry
        self.memory = original_agent.memory
        self.state_manager = original_agent.state_manager
        self.planner = original_agent.planner
    
    def process_query(self, query: str) -> str:
        """
        Process query with better handling for missing data.
        """
        print(f"\nProcessing: {query}")
        print("-" * 40)
        
        # Check if it's a calculator query
        if any(kw in query.lower() for kw in ['calculate', 'compute', 'what is', '+', '-', '*', '/']):
            # Try to extract calculation
            import re
            calc_match = re.search(r'([\d\s\+\-\*/()]+)', query)
            if calc_match:
                expr = calc_match.group(1).strip()
                if expr:
                    result = self.registry.execute_tool("calculator", expression=expr)
                    if result.get('success'):
                        return f"Calculation: {expr} = {result['result']}"
        
        # Try Wikipedia search
        wiki_result = self.registry.execute_tool("wikipedia_search", query=query)
        if wiki_result.get('success') and wiki_result.get('result'):
            if "No Wikipedia article" not in wiki_result['result']:
                return wiki_result['result']
        
        # Try web search as fallback
        web_result = self.registry.execute_tool("web_search", query=query)
        if web_result.get('success') and web_result.get('result'):
            return web_result['result']
        
        # Generic response when no data found
        return f"""
Information about '{query}':

I searched for information but couldn't find specific results. 
Here are some suggestions:
1. Try a simpler query (e.g., "AI" instead of "artificial intelligence and its applications")
2. Check spelling of your query
3. Try a different topic

For this query, you might want to:
- Check Wikipedia directly for "{query}"
- Use a search engine for more detailed information
- Break down your question into smaller parts
"""

# Create updated agent
updated_agent = UpdatedPlanningAgent(agent)

print("\n" + "=" * 60)
print("Updated agent created with better handling.")
print("=" * 60)

# Test the updated agent
print("\nTesting updated agent:")
result = updated_agent.process_query("rag model")
print(result)

print("\nTesting calculator:")
result = updated_agent.process_query("calculate 100/4")
print(result)


Updated agent created with better handling.

Testing updated agent:

Processing: rag model
----------------------------------------
Wikipedia Results:

Retrieval-augmented generation: generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources. With...
Large language model: (RAG), fine-tuning, and other methods. The matter of LLM&#039;s exhibiting intelligence or understanding has two main aspects—the first is how to mode...

Testing calculator:

Processing: calculate 100/4
----------------------------------------
Calculation: 100/4 = Result: 25.0


In [18]:
# CELL 13: Final Gradio Interface (Simple and Clean)
# This cell provides a clean, working Gradio interface

import gradio as gr

print("Building Final Agent Interface...")

class FinalInterface:
    def __init__(self, agent):
        self.agent = agent
        self.history = []
    
    def respond(self, message, history):
        if not message or message.strip() == "":
            return history + [(message, "Please enter a query.")]
        
        try:
            response = self.agent.process_query(message)
            history.append((message, response))
            return history
        except Exception as e:
            history.append((message, f"Error: {str(e)}"))
            return history

final_interface = FinalInterface(updated_agent)

with gr.Blocks(title="Agent Demo") as demo:
    gr.Markdown("""
    # Simple Agent Demo
    
    **What it can do:**
    - Calculate math expressions (e.g., "calculate 100/4")
    - Search Wikipedia for information
    - Provide helpful responses
    
    **Try:**
    - "calculate 25 * 4 + 10"
    - "tell me about artificial intelligence"
    - "rag model" (will give helpful suggestions)
    """)
    
    chatbot = gr.Chatbot(height=400)
    
    with gr.Row():
        msg = gr.Textbox(placeholder="Type your query here...", scale=8)
        submit = gr.Button("Send", variant="primary", scale=1)
    
    clear = gr.Button("Clear")
    
    def respond_to_message(message, history):
        return final_interface.respond(message, history), ""
    
    submit.click(respond_to_message, [msg, chatbot], [chatbot, msg])
    msg.submit(respond_to_message, [msg, chatbot], [chatbot, msg])
    clear.click(lambda: ([], ""), None, [chatbot, msg])

print("\nLaunching interface...")
demo.launch(share=True)

print("\n" + "=" * 60)
print("Day 2 Complete!")
print("=" * 60)

Building Final Agent Interface...

Launching interface...
* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://6b4a39876daf45b535.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



Day 2 Complete!

Processing: compare llama and gpt
----------------------------------------


In [19]:
# CELL 14: LLM Integrated Agent - Complete Version
# This cell creates an agent that uses LLM to answer ANY query

import os
import json
import requests
from datetime import datetime
from typing import Dict, Any, Optional, List

print("=" * 60)
print("BUILDING LLM-INTEGRATED AGENT")
print("=" * 60)

# ============================================================================
# PART 1: LLM Integration (Free API Options)
# ============================================================================

class LLMClient:
    """
    Client for interacting with LLM APIs.
    Supports multiple free/paid options.
    """
    
    def __init__(self, provider="huggingface"):
        """
        Initialize LLM client.
        
        Providers:
        - huggingface: Free tier (requires token)
        - together: Free tier
        - openai: Paid (requires API key)
        - groq: Free tier (fast)
        """
        self.provider = provider
        self.api_key = None
        
        # Try to get API keys from environment
        self.huggingface_token = os.environ.get('HUGGINGFACE_TOKEN', '')
        self.openai_key = os.environ.get('OPENAI_API_KEY', '')
        self.groq_key = os.environ.get('GROQ_API_KEY', '')
        self.together_key = os.environ.get('TOGETHER_API_KEY', '')
        
        print(f"LLM Client initialized with provider: {provider}")
    
    def generate_response(self, query: str, context: str = "") -> str:
        """
        Generate a response using LLM.
        
        Args:
            query: User query
            context: Additional context
            
        Returns:
            Generated response
        """
        # Try multiple providers in order
        providers = [
            ("huggingface", self._query_huggingface),
            ("groq", self._query_groq),
            ("together", self._query_together),
            ("openai", self._query_openai)
        ]
        
        for provider_name, provider_func in providers:
            try:
                print(f"  Trying {provider_name}...")
                response = provider_func(query, context)
                if response and len(response) > 50:
                    return response
            except Exception as e:
                print(f"  {provider_name} failed: {str(e)}")
                continue
        
        # Fallback: Local response generation
        return self._generate_fallback_response(query)
    
    def _query_huggingface(self, query: str, context: str) -> str:
        """Query Hugging Face Inference API (Free)."""
        if not self.huggingface_token:
            # Try without token (rate limited)
            pass
        
        # Use a free model
        model = "microsoft/DialoGPT-medium"
        url = f"https://api-inference.huggingface.co/models/{model}"
        
        headers = {"Authorization": f"Bearer {self.huggingface_token}"} if self.huggingface_token else {}
        
        payload = {
            "inputs": query,
            "parameters": {
                "max_length": 300,
                "temperature": 0.7
            }
        }
        
        response = requests.post(url, headers=headers, json=payload, timeout=15)
        
        if response.status_code == 200:
            data = response.json()
            if isinstance(data, list) and data:
                return data[0].get('generated_text', '')
        return ""
    
    def _query_groq(self, query: str, context: str) -> str:
        """Query Groq API (Free, fast)."""
        if not self.groq_key:
            return ""
        
        url = "https://api.groq.com/openai/v1/chat/completions"
        headers = {
            "Authorization": f"Bearer {self.groq_key}",
            "Content-Type": "application/json"
        }
        
        payload = {
            "model": "mixtral-8x7b-32768",
            "messages": [
                {"role": "system", "content": "You are a helpful AI assistant. Provide accurate and detailed responses."},
                {"role": "user", "content": query}
            ],
            "temperature": 0.7,
            "max_tokens": 500
        }
        
        response = requests.post(url, headers=headers, json=payload, timeout=15)
        
        if response.status_code == 200:
            data = response.json()
            return data['choices'][0]['message']['content']
        return ""
    
    def _query_together(self, query: str, context: str) -> str:
        """Query Together API (Free tier)."""
        if not self.together_key:
            return ""
        
        url = "https://api.together.xyz/v1/chat/completions"
        headers = {
            "Authorization": f"Bearer {self.together_key}",
            "Content-Type": "application/json"
        }
        
        payload = {
            "model": "mistralai/Mixtral-8x7B-Instruct-v0.1",
            "messages": [
                {"role": "system", "content": "You are a helpful AI assistant."},
                {"role": "user", "content": query}
            ],
            "temperature": 0.7,
            "max_tokens": 500
        }
        
        response = requests.post(url, headers=headers, json=payload, timeout=15)
        
        if response.status_code == 200:
            data = response.json()
            return data['choices'][0]['message']['content']
        return ""
    
    def _query_openai(self, query: str, context: str) -> str:
        """Query OpenAI API (Paid)."""
        if not self.openai_key:
            return ""
        
        url = "https://api.openai.com/v1/chat/completions"
        headers = {
            "Authorization": f"Bearer {self.openai_key}",
            "Content-Type": "application/json"
        }
        
        payload = {
            "model": "gpt-3.5-turbo",
            "messages": [
                {"role": "system", "content": "You are a helpful AI assistant."},
                {"role": "user", "content": query}
            ],
            "temperature": 0.7,
            "max_tokens": 500
        }
        
        response = requests.post(url, headers=headers, json=payload, timeout=15)
        
        if response.status_code == 200:
            data = response.json()
            return data['choices'][0]['message']['content']
        return ""
    
    def _generate_fallback_response(self, query: str) -> str:
        """
        Generate a response without API (knowledge base fallback).
        """
        # Basic knowledge base for common queries
        knowledge = {
            "llm": """Large Language Model (LLM) is a type of AI model trained on vast amounts of text data. 
Key points:
- Trained on billions of parameters
- Can understand and generate human-like text
- Examples: GPT, Claude, LLaMA, Gemini
- Applications: Chatbots, Content Creation, Code Generation""",

            "rag": """RAG (Retrieval-Augmented Generation) is a technique that:
1. Retrieves relevant information from a knowledge base
2. Augments the prompt with retrieved context
3. Generates response using LLM with context

Key components:
- Vector Database (ChromaDB, Pinecone)
- Embedding Models
- LLM (GPT, Claude)
- Document Store""",

            "transformer": """Transformer is a neural network architecture that:
- Uses self-attention mechanism
- Processes sequences in parallel
- Foundation of modern LLMs
- Key components: Encoder, Decoder, Attention layers""",

            "gpt": """GPT (Generative Pre-trained Transformer) is:
- Developed by OpenAI
- Trained on internet text
- Capable of: Text generation, Translation, Question answering
- Versions: GPT-1, GPT-2, GPT-3, GPT-4"""
        }
        
        query_lower = query.lower()
        for key, value in knowledge.items():
            if key in query_lower:
                return value
        
        return f"""I don't have specific information about '{query}' in my knowledge base.

To get detailed answers, you can:
1. **Get an API Key**: Sign up for free API keys:
   - Groq: https://console.groq.com (Free, fast)
   - Together AI: https://together.ai (Free tier)
   - Hugging Face: https://huggingface.co (Free tier)

2. **Set Environment Variable**:
   export GROQ_API_KEY="your_key_here"

3. **Ask me again** after setting up the API key.

Would you like me to help you set up any of these APIs?"""

def create_llm_client(provider: str = "huggingface") -> LLMClient:
    """Factory function to create LLM client."""
    return LLMClient(provider)

# ============================================================================
# PART 2: LLM-Integrated Agent
# ============================================================================

class LLMAgent:
    """
    Complete agent with LLM integration.
    Can answer ANY query using LLM or fallback.
    """
    
    def __init__(self, registry, memory_system, llm_client):
        self.registry = registry
        self.memory = memory_system
        self.llm = llm_client
        self.conversation_history = []
        
        print(f"\nLLM Agent initialized")
        print(f"  - Tools: {registry.list_tools()}")
        print(f"  - LLM Provider: {llm_client.provider}")
    
    def process_query(self, query: str) -> str:
        """
        Process ANY query using LLM.
        """
        print(f"\nProcessing: {query}")
        print("-" * 40)
        
        # Check if it's a math query (handle locally)
        import re
        calc_match = re.search(r'([\d\s\+\-\*/\(\)]+)', query)
        if calc_match and len(calc_match.group(1).strip()) > 2:
            expr = calc_match.group(1).strip()
            if not any(c.isalpha() for c in expr.replace('sqrt', '').replace('sin', '').replace('cos', '')):
                result = self.registry.execute_tool("calculator", expression=expr)
                if result.get('success'):
                    response = f"Calculation: {expr} = {result['result']}"
                    self._store_conversation(query, response)
                    return response
        
        # Try Wikipedia for informational queries
        if len(query.split()) < 5:  # Short queries might be Wikipedia topics
            wiki_result = self.registry.execute_tool("wikipedia_search", query=query)
            if wiki_result.get('success') and "No Wikipedia" not in wiki_result.get('result', ''):
                response = wiki_result['result']
                self._store_conversation(query, response)
                return response
        
        # Use LLM for everything else
        print("  Using LLM to generate response...")
        try:
            response = self.llm.generate_response(query)
            
            if not response or len(response) < 20:
                # If LLM fails, use fallback
                response = self.llm._generate_fallback_response(query)
            
            # Store in memory
            self.memory.store(f"Q: {query[:50]}... A: {response[:100]}...", "short_term", 5)
            self._store_conversation(query, response)
            
            return response
            
        except Exception as e:
            error_response = f"Error generating response: {str(e)}\n\n{self.llm._generate_fallback_response(query)}"
            return error_response
    
    def _store_conversation(self, query: str, response: str):
        """Store conversation in history."""
        self.conversation_history.append({
            "timestamp": datetime.now().isoformat(),
            "query": query,
            "response_preview": response[:200]
        })

def create_llm_agent(registry, memory, llm_client) -> LLMAgent:
    """Factory function to create LLM agent."""
    return LLMAgent(registry, memory, llm_client)

# ============================================================================
# PART 3: Build and Test
# ============================================================================

print("\n" + "=" * 60)
print("BUILDING LLM AGENT")
print("=" * 60)

# Create LLM client
llm_client = create_llm_client("huggingface")

# Create LLM agent
llm_agent = create_llm_agent(registry, memory, llm_client)

# Test the agent
print("\n" + "=" * 60)
print("TESTING LLM AGENT")
print("=" * 60)

test_queries = [
    "what is llm",
    "what is rag model",
    "explain transformer architecture",
    "what is the difference between gpt and bert",
    "calculate 100 * 25"
]

for query in test_queries:
    print("\n" + "-" * 40)
    response = llm_agent.process_query(query)
    print(f"\nResponse:\n{response[:500]}...")

print("\n" + "=" * 60)
print("LLM AGENT READY")
print("=" * 60)

# ============================================================================
# PART 4: Gradio Interface
# ============================================================================

import gradio as gr

class LLMInterface:
    def __init__(self, agent):
        self.agent = agent
    
    def respond(self, message, history):
        if not message or message.strip() == "":
            return history + [(message, "Please enter a query.")]
        
        try:
            response = self.agent.process_query(message)
            history.append((message, response))
            return history
        except Exception as e:
            history.append((message, f"Error: {str(e)}"))
            return history

interface = LLMInterface(llm_agent)

with gr.Blocks(title="LLM Agent - Ask Anything!", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # LLM Agent - Ask Anything!
    
    This agent uses LLM to answer ANY query. Try asking:
    - "what is artificial intelligence"
    - "explain quantum computing"
    - "what is the difference between machine learning and deep learning"
    - "calculate 100/4"
    - "tell me about RAG model"
    """)
    
    chatbot = gr.Chatbot(height=500)
    
    with gr.Row():
        msg = gr.Textbox(placeholder="Ask anything...", scale=8)
        submit = gr.Button("Send", variant="primary", scale=1)
    
    clear = gr.Button("Clear Conversation")
    
    def respond_to_message(message, history):
        return interface.respond(message, history), ""
    
    submit.click(respond_to_message, [msg, chatbot], [chatbot, msg])
    msg.submit(respond_to_message, [msg, chatbot], [chatbot, msg])
    clear.click(lambda: ([], ""), None, [chatbot, msg])

print("\n" + "=" * 60)
print("Launching Gradio Interface...")
print("=" * 60)
print("\n✅ Now you can ask ANY question!")
print("   The agent will try multiple LLM providers:")
print("   1. Hugging Face (Free)")
print("   2. Groq (Free, fast)")
print("   3. Together AI (Free)")
print("   4. OpenAI (Paid)")
print("   5. Fallback knowledge base")
print("\n" + "=" * 60)

demo.launch(share=True)

BUILDING LLM-INTEGRATED AGENT

BUILDING LLM AGENT
LLM Client initialized with provider: huggingface

LLM Agent initialized
  - Tools: ['wikipedia_search', 'web_search', 'calculator']
  - LLM Provider: huggingface

TESTING LLM AGENT

----------------------------------------

Processing: what is llm
----------------------------------------

Response:
Wikipedia Results:

Large language model: A large language model (LLM) is an AI model (typically a neural network) trained on a vast amount of text for natural language processing tasks, espec...
List of large language models: language model (LLM) is a type of machine learning model designed for natural language processing tasks such as language generation. LLMs are language......

----------------------------------------

Processing: what is rag model
----------------------------------------

Response:
Wikipedia Results:

Retrieval-augmented generation: generation (RAG) is a technique that enables large language models (LLMs) to retrieve an